In [1]:
!pip install librosa soundfile tqdm --quiet

In [2]:
import os
import random
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import zipfile
import io
from tqdm import tqdm

In [3]:
# --- CONFIGURATION ---
ICBHI_PATH = "/kaggle/input/datasets/husninm/icbhi-2017-challenge/ICBHI_final_database"
LUNG_SOUNDS_PATH = "/kaggle/input/datasets/bytevance/lung-sounds/Audio Files"
BREATH_PATH = "/kaggle/input/datasets/mathias256/breath/BREATH"   

SNR_RANGE = [0, 5, 10, 15]
OUTPUT_DIR = "/kaggle/working/tse_dataset_split"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
num_samples = 920
train_count = int(0.7 * num_samples)
val_count = int(0.2 * num_samples)
test_count = num_samples - train_count - val_count

def get_wav_files(directory):
    # We need to loop through 'files' to define 'f'
    return [os.path.join(root, f) for root, _, files in os.walk(directory) for f in files if f.endswith(".wav")]

In [5]:
# Load lists
icbhi_list = get_wav_files(ICBHI_PATH)
breath_list = get_wav_files(BREATH_PATH)
lung_list = get_wav_files(LUNG_SOUNDS_PATH)

print(f"Sources: ICBHI({len(icbhi_list)}), Breath({len(breath_list)}), Lung({len(lung_list)})")

Sources: ICBHI(920), Breath(520), Lung(336)


In [6]:
def mix_three_datasets(s1, s2, s3, snr12, snr13):
    # 1. Remove DC Offset (prevents hum)
    s1 = s1 - np.mean(s1)
    s2 = s2 - np.mean(s2)
    s3 = s3 - np.mean(s3)

    target_len = len(s1)
    def pad_or_tile(sig, length):
        if len(sig) == 0: return np.zeros(length)
        if len(sig) < length:
            return np.tile(sig, int(np.ceil(length / len(sig))))[:length]
        return sig[:length]

    s2 = pad_or_tile(s2, target_len)
    s3 = pad_or_tile(s3, target_len)

    # 2. Add a sensible epsilon to avoid over-amplifying silence
    rms1 = np.sqrt(np.mean(s1**2)) + 1e-4 
    rms2 = np.sqrt(np.mean(s2**2)) + 1e-4
    rms3 = np.sqrt(np.mean(s3**2)) + 1e-4

    factor2 = 10**(snr12 / 20.0)
    factor3 = 10**(snr13 / 20.0)
    
    s2_scaled = s2 * (rms1 / (factor2 * rms2))
    s3_scaled = s3 * (rms1 / (factor3 * rms3))
    
    mixed = s1 + s2_scaled + s3_scaled

    # Peak normalization
    max_val = np.max(np.abs(mixed))
    if max_val > 1.0:
        mixed /= max_val
        s1 /= max_val
        s2_scaled /= max_val
        s3_scaled /= max_val

    return mixed, s1, s2_scaled, s3_scaled

In [7]:
# Split logic
indices = list(range(num_samples))
random.shuffle(indices)
split_map = {idx: ("train" if i < train_count else "eval" if i < train_count + val_count else "test") 
             for i, idx in enumerate(indices)}

# Prepare Zips
zip_names = [f"{s}_{t}" for s in ["train", "eval", "test"] for t in ["mix", "s1", "s2", "s3"]]
zips = {k: zipfile.ZipFile(os.path.join(OUTPUT_DIR, f"{k}.zip"), 'w') for k in zip_names}
metadata = []

In [8]:
# Processing
for idx in tqdm(range(num_samples)):
    split = split_map[idx]
    
    # Using random.choice handles the different list lengths automatically
    s1_path = random.choice(icbhi_list)
    s2_path = random.choice(breath_list)
    s3_path = random.choice(lung_list)

    s1_audio, _ = librosa.load(s1_path, sr=16000)
    s2_audio, _ = librosa.load(s2_path, sr=16000)
    s3_audio, _ = librosa.load(s3_path, sr=16000)

    snr12, snr13 = random.choice(SNR_RANGE), random.choice(SNR_RANGE)
    mixed, s1_f, s2_f, s3_f = mix_three_datasets(s1_audio, s2_audio, s3_audio, snr12, snr13)

    base_name = f"sample_{idx}"
    audios = {"mix": mixed, "s1": s1_f, "s2": s2_f, "s3": s3_f}

    for key, data in audios.items():
        fname = f"{base_name}_{key}.wav"
        buf = io.BytesIO()
        sf.write(buf, data, 16000, format='WAV')
        zips[f"{split}_{key}"].writestr(fname, buf.getvalue())

    metadata.append({
        "ID": idx, "split": split, "mixture_wav": f"{base_name}_mix.wav",
        "s1_wav": f"{base_name}_s1.wav", "s2_wav": f"{base_name}_s2.wav", "s3_wav": f"{base_name}_s3.wav",
        "snr_1_2": snr12, "snr_1_3": snr13
    })

for z in zips.values(): z.close()
pd.DataFrame(metadata).to_csv(os.path.join(OUTPUT_DIR, "metadata_split.csv"), index=False)

100%|██████████| 920/920 [01:22<00:00, 11.17it/s]
